## PHASE 0 – Load & hiểu data

In [2]:
# Visualization
import matplotlib.pyplot as plt

# Data processing
import pandas as pd

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score

# CatBoost
from catboost import CatBoostClassifier, Pool, cv

In [3]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

Số dòng, số cột

In [4]:
print("số dòng:", len(df))
print("số cột:", len(df.columns))

số dòng: 1470
số cột: 35


In [5]:
df['Attrition'].value_counts(normalize=True)

Attrition
No     0.838776
Yes    0.161224
Name: proportion, dtype: float64

**Nhận thấy: Dữ liệu bị mất cân bằng**

Kiểm tra missing value

In [6]:
df.isna().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

## PHASE 1 – Xác định feature types

Drop các feature vô nghĩa

In [7]:
drop_cols = [
    'EmployeeNumber',   # ID
    'EmployeeCount',    # constant
    'Over18',           # constant
    'StandardHours'     # constant
]
df = df.drop(columns=drop_cols)

In [8]:
df['Attrition'] = df['Attrition'].map({'No': 0, 'Yes': 1})

In [9]:
df.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField',
       'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus',
       'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
       'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager'],
      dtype='object')

In [10]:
df.head().T

,0,1,2,3,4
Age,41,49,37,33,27
Attrition,1,0,1,0,0
BusinessTravel,Travel_Rarely,Travel_Frequently,Travel_Rarely,Travel_Frequently,Travel_Rarely
DailyRate,1102,279,1373,1392,591
Department,Sales,Research & Development,Research & Development,Research & Development,Research & Development
DistanceFromHome,1,8,2,3,2
Education,2,1,2,4,1
EducationField,Life Sciences,Life Sciences,Other,Life Sciences,Medical
EnvironmentSatisfaction,2,3,4,4,1
Gender,Female,Male,Male,Female,Male


In [11]:
cat_features = [
    'BusinessTravel',
    'Department',
    'EducationField',
    'Gender',
    'JobRole',
    'MaritalStatus',
    'OverTime'
]

## PHASE 2 – Split dữ liệu
Chia 80/20

In [12]:
X = df.drop(columns='Attrition')
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print("Số lượng mẫu training:", len(y_train))
print("Số lượng mẫu testing:", len(y_test))

Số lượng mẫu training: 1176
Số lượng mẫu testing: 294


## PHASE 3 – Pool cho CatBoost

In [13]:
train_pool = Pool(
    X_train, y_train,
    cat_features=cat_features
)

test_pool = Pool(
    X_test, y_test,
    cat_features=cat_features
)

## PHASE 4 – Train baseline CatBoost (KHÔNG TUNING)

In [14]:
baseline_model = CatBoostClassifier(
    loss_function='Logloss',
    iterations=100,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=False
)

baseline_model.fit(
    train_pool,
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [15]:
proba_test = baseline_model.predict_proba(X_test)[:,1]
auc_test = roc_auc_score(y_test, proba_test)

train_pred = baseline_model.predict_proba(X_train)[:,1]
auc_train = roc_auc_score(y_train, train_pred)

gap = auc_train - auc_test

print(f"Train AUC: {auc_train:.4f}")
print(f"Test AUC:  {auc_test:.4f}")
print(f"Gap:       {gap:.4f}")

Train AUC: 0.9979
Test AUC:  0.7890
Gap:       0.2089


## Phrase 5 - Cross - Validation

In [16]:
cv_params = {
    'iterations': 1000,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'early_stopping_rounds': 50,
}

cv_results = cv(
    pool=train_pool,
    params=cv_params,
    fold_count=5,
    shuffle=True,
    partition_random_seed=42,
    plot=True,
    verbose=False,
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Training on fold [0/5]

bestTest = 0.812466773
bestIteration = 23

Training on fold [1/5]

bestTest = 0.8123163238
bestIteration = 116

Training on fold [2/5]

bestTest = 0.7686347849
bestIteration = 37

Training on fold [3/5]

bestTest = 0.8157894737
bestIteration = 151

Training on fold [4/5]

bestTest = 0.8589366818
bestIteration = 101



In [17]:
best_iter = cv_results['test-AUC-mean'].idxmax()
best_auc = cv_results['test-AUC-mean'].max()

print(f"Best iteration: {best_iter}")
print(f"Best CV AUC: {best_auc:.4f}")

Best iteration: 150
Best CV AUC: 0.8086


## Phrase 6 - Grid Search

In [34]:
base_model = CatBoostClassifier(
    iterations = best_iter + 30,
    loss_function='Logloss',
    auto_class_weights='Balanced',
    eval_metric='AUC',
    random_seed=42,
    task_type='GPU',
    verbose=20,
)

#### Tuning 

##### Lần 1

In [39]:
param_grid = {
    "depth": [2, 3, 5],
    "learning_rate": [0.03, 0.05, 0.1],
    "l2_leaf_reg": [20, 30, 40]
}

gs = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    return_train_score=True,
)

gs.fit(X_train, y_train, cat_features=cat_features)

results = pd.DataFrame(gs.cv_results_)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 17.5ms	remaining: 3.13s
20:	total: 339ms	remaining: 2.56s
40:	total: 645ms	remaining: 2.19s
60:	total: 954ms	remaining: 1.86s
80:	total: 1.26s	remaining: 1.54s
100:	total: 1.57s	remaining: 1.23s
120:	total: 1.87s	remaining: 914ms
140:	total: 2.19s	remaining: 607ms
160:	total: 2.51s	remaining: 296ms
179:	total: 2.8s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 17.8ms	remaining: 3.19s
20:	total: 319ms	remaining: 2.41s
40:	total: 643ms	remaining: 2.18s
60:	total: 960ms	remaining: 1.87s
80:	total: 1.27s	remaining: 1.55s
100:	total: 1.56s	remaining: 1.22s
120:	total: 1.89s	remaining: 921ms
140:	total: 2.2s	remaining: 609ms
160:	total: 2.5s	remaining: 296ms
179:	total: 2.81s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


...

0:	total: 21ms	remaining: 3.76s
20:	total: 383ms	remaining: 2.9s
40:	total: 748ms	remaining: 2.53s
60:	total: 1.11s	remaining: 2.17s
80:	total: 1.47s	remaining: 1.8s
100:	total: 1.85s	remaining: 1.45s
120:	total: 2.22s	remaining: 1.08s
140:	total: 2.59s	remaining: 716ms
160:	total: 2.97s	remaining: 351ms
179:	total: 3.32s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 24.9ms	remaining: 4.45s
20:	total: 399ms	remaining: 3.02s
40:	total: 773ms	remaining: 2.62s
60:	total: 1.16s	remaining: 2.27s
80:	total: 1.53s	remaining: 1.87s
100:	total: 1.91s	remaining: 1.49s
120:	total: 2.28s	remaining: 1.11s
140:	total: 2.64s	remaining: 731ms
160:	total: 3.03s	remaining: 358ms
179:	total: 3.41s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 17.4ms	remaining: 3.11s
20:	total: 337ms	remaining: 2.56s
40:	total: 647ms	remaining: 2.19s
60:	total: 950ms	remaining: 1.85s
80:	total: 1.26s	remaining: 1.54s
100:	total: 1.57s	remaining: 1.23s
120:	total: 1.92s	remaining: 937ms
140:	total: 2.33s	remaining: 644ms
160:	total: 2.69s	remaining: 318ms
179:	total: 3.04s	remaining: 0us


In [44]:
results["gap"] = results["mean_train_score"] - results["mean_test_score"]
params_df = results["params"].apply(pd.Series)

final_df = pd.concat([
    params_df,
    results[["mean_train_score","mean_test_score","gap"]]
], axis=1)

final_df.sort_values(
    ["mean_test_score","gap"],
    ascending=[False,True]
).head(10)

,depth,l2_leaf_reg,learning_rate,mean_train_score,mean_test_score,gap
7,2.0,40.0,0.05,0.920599,0.809685,0.110914
4,2.0,30.0,0.05,0.920818,0.808679,0.112139
1,2.0,20.0,0.05,0.922097,0.808519,0.113578
8,2.0,40.0,0.10,0.953657,0.807898,0.145759
10,3.0,20.0,0.05,0.963305,0.807830,0.155475
13,3.0,30.0,0.05,0.960202,0.807750,0.152452
16,3.0,40.0,0.05,0.957427,0.807549,0.149878
17,3.0,40.0,0.10,0.986296,0.807096,0.179199
14,3.0,30.0,0.10,0.988428,0.806761,0.181667
6,2.0,40.0,0.03,0.893576,0.806688,0.086888


**Nhận xét: depth = 2, lr = 0.05 tốt rõ rệt, l2_leaf_reg càng lớn càng tốt => tiếp tục tuning l2_leaf_reg, thử các lân cận của lr**

##### Lần 2

In [52]:
param_grid = {
    "depth": [2],
    "learning_rate": [0.03, 0.04, 0.05],
    "l2_leaf_reg": [40, 50, 60, 70],
}
gs1 = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    return_train_score=True,
)

gs1.fit(X_train, y_train, cat_features=cat_features)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 19.1ms	remaining: 3.42s
20:	total: 367ms	remaining: 2.78s
40:	total: 744ms	remaining: 2.52s
60:	total: 1.11s	remaining: 2.16s
80:	total: 1.44s	remaining: 1.76s
100:	total: 1.79s	remaining: 1.4s
120:	total: 2.15s	remaining: 1.05s
140:	total: 2.51s	remaining: 695ms
160:	total: 2.82s	remaining: 332ms
179:	total: 3.15s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 21.8ms	remaining: 3.9s
20:	total: 391ms	remaining: 2.96s
40:	total: 758ms	remaining: 2.57s
60:	total: 1.09s	remaining: 2.12s
80:	total: 1.4s	remaining: 1.71s
100:	total: 1.71s	remaining: 1.34s
120:	total: 2.03s	remaining: 990ms
140:	total: 2.35s	remaining: 649ms
160:	total: 2.67s	remaining: 315ms
179:	total: 2.98s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


...

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 17.8ms	remaining: 3.19s
20:	total: 342ms	remaining: 2.59s
40:	total: 680ms	remaining: 2.31s
60:	total: 1s	remaining: 1.96s
80:	total: 1.32s	remaining: 1.62s
100:	total: 1.66s	remaining: 1.3s
120:	total: 2.07s	remaining: 1.01s
140:	total: 2.42s	remaining: 671ms
160:	total: 2.74s	remaining: 324ms
179:	total: 3.05s	remaining: 0us


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 18.5ms	remaining: 3.31s
20:	total: 333ms	remaining: 2.52s
40:	total: 648ms	remaining: 2.2s
60:	total: 1.03s	remaining: 2.02s
80:	total: 1.41s	remaining: 1.72s
100:	total: 1.76s	remaining: 1.38s
120:	total: 2.14s	remaining: 1.04s
140:	total: 2.5s	remaining: 692ms
160:	total: 2.86s	remaining: 337ms
179:	total: 3.19s	remaining: 0us


,estimator,<catboost.cor...00194EDF55C00>
,param_grid,"{'depth': [2], 'l2_leaf_reg': [40, 50, ...], 'learning_rate': [0.03, 0.04, ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True


In [53]:
results = pd.DataFrame(gs1.cv_results_)
results["gap"] = results["mean_train_score"] - results["mean_test_score"]
params_df = results["params"].apply(pd.Series)

final_df = pd.concat([
    params_df,
    results[["mean_train_score","mean_test_score","gap"]]
], axis=1)

final_df.sort_values(
    ["mean_test_score","gap"],
    ascending=[False,True]
).head(10)

,depth,l2_leaf_reg,learning_rate,mean_train_score,mean_test_score,gap
4,2.0,50.0,0.04,0.907350,0.812653,0.094698
1,2.0,40.0,0.04,0.907861,0.812040,0.095821
8,2.0,60.0,0.05,0.917999,0.810544,0.107455
7,2.0,60.0,0.04,0.905983,0.810516,0.095466
10,2.0,70.0,0.04,0.905415,0.810459,0.094956
11,2.0,70.0,0.05,0.917146,0.810219,0.106928
2,2.0,40.0,0.05,0.920599,0.809685,0.110914
5,2.0,50.0,0.05,0.919334,0.809661,0.109672
9,2.0,70.0,0.03,0.891021,0.807914,0.083107
3,2.0,50.0,0.03,0.893049,0.806951,0.086098


| Metric    | Trước tuning | Sau tuning | Ý nghĩa |
|-----------|--------------|------------|---------|
| Train AUC | 0.9979 | 0.9073 | Giảm overfitting |
| Test AUC  | 0.7890 | 0.8127 | Khả năng dự đoán tốt hơn |
| Gap       | 0.2089 | 0.0947 | Generalization tốt hơn |

Sau quá trình hyperparameter tuning và cross-validation với CatBoost, hiệu năng mô hình đã được cải thiện:

- Test AUC tăng từ 0.7890 lên 0.8127 (+0.0237), cho thấy khả năng dự đoán trên dữ liệu chưa thấy được cải thiện.

- Train AUC giảm từ 0.9979 xuống 0.9073, cho thấy mô hình đã giảm hiện tượng overfitting.

- Khoảng cách Train–Test (Gap) giảm từ 0.2089 xuống 0.0947, chứng tỏ khả năng tổng quát hóa của mô hình tốt hơn.

Cấu hình hyperparameter tốt nhất được tìm thấy:
- `depth = 4`
- `learning_rate = 0.04`
- `l2_leaf_reg = 50`  
=> Với cấu hình này, mô hình đạt Test AUC ≈ 0.81, cùng với Train–Test gap < 0.1, cho thấy mức độ overfitting đã được kiểm soát và mô hình có khả năng dự đoán ổn định hơn trên dữ liệu kiểm thử.

#### Train lại model với best params + full train data

In [61]:
best_model = CatBoostClassifier(
    depth=2,
    learning_rate=0.04,
    l2_leaf_reg=50,
    auto_class_weights='Balanced',
    iterations=best_iter,
    eval_metric="AUC",
    task_type='GPU',
    verbose=50
)

best_model.fit(train_pool)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 17.3ms	remaining: 2.58s
50:	total: 936ms	remaining: 1.82s
100:	total: 1.87s	remaining: 907ms
149:	total: 2.8s	remaining: 0us


## PHASE 5 – Đánh giá sau khi train lại model với full data

In [62]:
# Train
train_pred = best_model.predict_proba(train_pool)[:,1]
train_auc = roc_auc_score(y_train, train_pred)

# Test
test_pred = best_model.predict_proba(test_pool)[:,1]
test_auc = roc_auc_score(y_test, test_pred)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)
print("Gap:", train_auc - test_auc)

Train AUC: 0.8901462581402797
Test AUC: 0.8008441726246878
Gap: 0.08930208551559193


In [63]:
from sklearn.metrics import classification_report

y_pred = best_model.predict(test_pool)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.79      0.86       247
           1       0.41      0.74      0.53        47

    accuracy                           0.79       294
   macro avg       0.67      0.77      0.69       294
weighted avg       0.86      0.79      0.81       294



### Đánh giá mô hình (CatBoost)

Sau khi thực hiện tuning tham số cho mô hình CatBoost, hiệu năng mô hình đã được cải thiện rõ rệt.

| Metric | Trước tuning | Sau tuning |
|------|------|------|
| Train AUC | 0.9979 | 0.89014 |
| Test AUC | 0.7890 | 0.80084 |
| Gap | 0.2089 | 0.089 |

#### Nhận xét

- **Overfitting giảm đáng kể**: Ban đầu mô hình có Train AUC gần 1 (0.9979) nhưng Test AUC thấp hơn nhiều, dẫn đến gap lớn (~0.21), cho thấy mô hình bị overfitting.
- **Khả năng tổng quát hóa được cải thiện**: Sau khi tuning, gap giảm xuống còn ~0.09, cho thấy mô hình tổng quát hóa tốt hơn trên dữ liệu chưa thấy.
- **Hiệu năng trên test set tăng**: Test AUC tăng từ **0.789 → 0.80084**, cho thấy các tham số mới giúp mô hình học được các pattern tốt hơn.
- **Train AUC giảm là hợp lý**: Train AUC giảm xuống 0.896 cho thấy mô hình không còn ghi nhớ dữ liệu train mà học các đặc trưng tổng quát hơn.

#### Kết luận

Việc tuning tham số đã giúp **giảm overfitting và cải thiện hiệu năng dự đoán của mô hình**, đạt được sự cân bằng tốt giữa khả năng học dữ liệu huấn luyện và khả năng tổng quát hóa.

## PHASE 6 – Giải thích model

In [64]:
importances = best_model.get_feature_importance()
features = X.columns

pd.Series(importances, index=features)\
  .sort_values(ascending=False)\
  .head(10)

OverTime                   16.468882
JobRole                    11.311574
StockOptionLevel            9.494780
YearsWithCurrManager        7.292331
Age                         7.154171
MonthlyIncome               5.630224
NumCompaniesWorked          5.624582
EnvironmentSatisfaction     4.893169
YearsAtCompany              4.668608
TotalWorkingYears           4.594638
dtype: float64

### Phân tích

- Kết quả feature importance từ CatBoost cho thấy các yếu tố liên quan đến điều kiện làm việc, thu nhập và kinh nghiệm nghề nghiệp có ảnh hưởng lớn đến dự đoán của mô hình.

- OverTime (16.47%) là yếu tố quan trọng nhất. Điều này cho thấy việc làm thêm giờ có ảnh hưởng mạnh đến khả năng xảy ra kết quả mục tiêu (ví dụ: nghỉ việc).

- JobRole (11.31%) đứng thứ hai, cho thấy vai trò công việc khác nhau có mức độ rủi ro khác nhau.

- StockOptionLevel (9.49%) cũng có ảnh hưởng đáng kể, cho thấy các chính sách cổ phiếu thưởng có thể tác động đến hành vi nhân viên.

- Các yếu tố liên quan đến kinh nghiệm và thời gian làm việc như YearsWithCurrManager, YearsAtCompany, và TotalWorkingYears cũng đóng vai trò quan trọng trong dự đoán của mô hình.

- Ngoài ra, các yếu tố cá nhân như Age và MonthlyIncome cũng góp phần đáng kể trong việc phân biệt các trường hợp.

- Nhìn chung, mô hình không chỉ dựa vào một biến duy nhất mà khai thác nhiều khía cạnh khác nhau của hồ sơ nhân viên, bao gồm điều kiện làm việc, kinh nghiệm và chế độ đãi ngộ.